<a href="https://colab.research.google.com/github/srinithiarulprakash/data-analysis/blob/main/online_retail_sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
data = pd.read_csv(r"/content/online_retail.csv")

In [9]:
df_clean = data[data['Quantity'] > 0]

In [10]:
df_clean1 = data[data['UnitPrice'] > 0]

In [11]:
df_clean2 = df_clean1.dropna(subset=['CustomerID'])

In [12]:
print(df_clean2.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER       6.0   
1    536365     71053                  WHITE METAL LANTERN       6.0   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER       8.0   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE       6.0   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.       6.0   

           InvoiceDate  UnitPrice  CustomerID         Country  
0  2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2  2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4  2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [13]:
print(df_clean)

       InvoiceNo StockCode                          Description  Quantity  \
0         536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER       6.0   
1         536365     71053                  WHITE METAL LANTERN       6.0   
2         536365    84406B       CREAM CUPID HEARTS COAT HANGER       8.0   
3         536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE       6.0   
4         536365    84029E       RED WOOLLY HOTTIE WHITE HEART.       6.0   
...          ...       ...                                  ...       ...   
187902    553011     23284        DOORMAT KEEP CALM AND COME IN       4.0   
187903    553012     22925        BLUE GIANT GARDEN THERMOMETER       4.0   
187904    553012     22927       GREEN GIANT GARDEN THERMOMETER       2.0   
187905    553012     22928      YELLOW GIANT GARDEN THERMOMETER       4.0   
187906    553012    84625C  BLUE NEW BAROQUE CANDLESTICK CANDLE      24.0   

                InvoiceDate  UnitPrice  CustomerID         Country  
0     

In [14]:
df_clean2['TotalSales'] = df_clean2['Quantity'] * df_clean2['UnitPrice']

/tmp/ipykernel_5916/3653552093.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean2['TotalSales'] = df_clean2['Quantity'] * df_clean2['UnitPrice']


In [17]:
df_clean['TotalSales'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

/tmp/ipykernel_5916/2895335506.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['TotalSales'] = df_clean['Quantity'] * df_clean['UnitPrice']
/tmp/ipykernel_5916/2895335506.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])


In [18]:
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

/tmp/ipykernel_5916/2237767826.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.to_period('M')
/tmp/ipykernel_5916/2237767826.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.day_name()
/tmp/ipykernel_5916/2237767826.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documenta

In [21]:
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceNo': 'nunique',                                    # Frequency
    'TotalSales': 'sum'                                        # Monetary
}).rename(columns={'InvoiceDate': 'Recency', 'InvoiceNo': 'Frequency', 'TotalSales': 'Monetary'})

In [23]:
# 2. Assign Quartile Scores (1 to 4)
# Note: For Recency, lower days = better score, so we reverse the labels [4, 3, 2, 1]
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4])

# Combine scores into a single RFM string
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

# 3. Define Customer Tiers based on scores
def segment_customer(df):
    if df['RFM_Score'] == '444':
        return 'VIP / Champions'
    elif int(df['F_Score']) >= 3 and int(df['M_Score']) >= 3:
        return 'Loyal Customers'
    elif int(df['R_Score']) >= 3 and int(df['F_Score']) == 1:
        return 'Recent / New Customers'
    elif int(df['R_Score']) <= 2 and int(df['F_Score']) >= 2:
        return 'At-Risk Customers'
    else:
        return 'Low-Value / Lost'

rfm['Customer_Segment'] = rfm.apply(segment_customer, axis=1)

# View the distribution of customer segments
print(rfm['Customer_Segment'].value_counts())

Customer_Segment
Low-Value / Lost          784
Loyal Customers           765
At-Risk Customers         567
VIP / Champions           252
Recent / New Customers    204
Name: count, dtype: int64


In [24]:
# Export your cleaned transaction data and customer segments
df_clean.to_csv('cleaned_online_retail.csv', index=False)
rfm.to_csv('customer_rfm_segments.csv', index=True)